# 03 — Error Analysis
Deep-dive into where the model fails: false positives, false negatives, and per-category confusion.

In [ ]:
import json
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from transformers import pipeline
from seqeval.metrics import classification_report

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

test = load_jsonl('data/annotated/test.jsonl')
print(f'Test samples: {len(test)}')

In [ ]:
# Load model
ner = pipeline(
    'token-classification',
    model='models/deberta-jd-bias-v1',
    aggregation_strategy='simple',
    device=0 if torch.cuda.is_available() else -1
)
print('Model loaded.')

In [ ]:
# Collect false positives and false negatives per category
false_positives = defaultdict(list)   # predicted bias, actually neutral
false_negatives = defaultdict(list)   # actual bias, not predicted

for sample in test:
    text   = sample['text']
    tokens = sample['tokens']
    true_labels = sample['labels']

    # True biased spans
    true_spans = set()
    for tok, lbl in zip(tokens, true_labels):
        if lbl.startswith('B-'):
            true_spans.add((tok.lower(), lbl[2:]))

    # Predicted spans
    pred_spans = set()
    for ent in ner(text):
        pred_spans.add((ent['word'].strip().lower(), ent['entity_group']))

    for span, cat in pred_spans - true_spans:
        false_positives[cat].append({'text': text[:80], 'phrase': span})

    for span, cat in true_spans - pred_spans:
        false_negatives[cat].append({'text': text[:80], 'phrase': span})

print('FP counts:', {k: len(v) for k, v in false_positives.items()})
print('FN counts:', {k: len(v) for k, v in false_negatives.items()})

In [ ]:
# Visualise FP vs FN per category
cats = ['GENDER_CODED', 'AGEIST', 'EXCLUSIONARY', 'ABILITY_CODED']
fp_counts = [len(false_positives[c]) for c in cats]
fn_counts = [len(false_negatives[c]) for c in cats]

x = range(len(cats))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar([i - 0.2 for i in x], fp_counts, 0.4, label='False positives', color='#F0997B')
ax.bar([i + 0.2 for i in x], fn_counts, 0.4, label='False negatives', color='#AFA9EC')
ax.set_xticks(x); ax.set_xticklabels(cats, rotation=15)
ax.set_ylabel('Count'); ax.legend(); ax.set_title('FP vs FN per bias category')
plt.tight_layout(); plt.show()

In [ ]:
# Inspect top false positives (model over-triggered)
print('=== Top False Positives ===')
for cat in cats:
    print(f'\n-- {cat} --')
    for ex in false_positives[cat][:3]:
        print(f"  Phrase: '{ex['phrase']}'")
        print(f"  Context: {ex['text']}...")

In [ ]:
# Inspect top false negatives (model missed these)
print('=== Top False Negatives (missed bias) ===')
for cat in cats:
    print(f'\n-- {cat} --')
    for ex in false_negatives[cat][:3]:
        print(f"  Phrase: '{ex['phrase']}'")
        print(f"  Context: {ex['text']}...")

In [ ]:
# Confidence distribution for correct vs incorrect predictions
correct_confs, wrong_confs = [], []

for sample in test:
    text = sample['text']
    true_labels = sample['labels']
    true_biased = any(l != 'O' for l in true_labels)

    for ent in ner(text):
        phrase = ent['word'].strip().lower()
        found  = any(t.lower() == phrase for t, l in zip(sample['tokens'], true_labels) if l != 'O')
        if found:
            correct_confs.append(ent['score'])
        else:
            wrong_confs.append(ent['score'])

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(correct_confs, bins=20, alpha=0.6, label='Correct predictions', color='#5DCAA5')
ax.hist(wrong_confs,   bins=20, alpha=0.6, label='Wrong predictions',   color='#F0997B')
ax.set_xlabel('Confidence'); ax.set_ylabel('Count')
ax.set_title('Confidence distribution: correct vs wrong predictions')
ax.legend(); plt.tight_layout(); plt.show()
print(f'Optimal threshold estimate: {sum(correct_confs)/len(correct_confs):.2f}')